## ============================================================================
## PHASE 4: FEDERATED LEARNING SETUP
### Federated Learning Breast Cancer Detection Project
## ============================================================================
### This notebook implements Federated Averaging (FedAvg) across 3 hospitals:
#### - Hospital 1 (WDBC) - Cell features
#### - Hospital 2 (Coimbra) - Blood biomarkers  
#### - Hospital 3 (BreakHis) - Histopathology images
##
### Key steps per round:
#### 1. Each hospital trains locally on their data
#### 2. Extract output layer weights from each hospital
#### 3. Server aggregates weights using weighted averaging
#### 4. Send global weights back to all hospitals
#### 5. Repeat for multiple rounds
## ============================================================================

### Setting Working Directory

In [1]:
# First cell in Colab
from google.colab import drive
drive.mount('/content/drive')

# Check GPU
import torch
print(f"GPU available : {torch.cuda.is_available()}")
print(f"GPU name      : {torch.cuda.get_device_name(0)}")
# Should print: Tesla T4 or similar

# Install dependencies
!pip install torchvision torch -q

Mounted at /content/drive
GPU available : True
GPU name      : Tesla T4


In [2]:
import os

curr_dir = os.getcwd()
print(f"Current directory: '{curr_dir}'")
os.chdir('/content/drive/MyDrive/BREAST_CANCER_FL/')
print("Changing to root directory...")

print(f"Working in: '{os.getcwd()}'")

Current directory: '/content'
Changing to root directory...
Working in: '/content/drive/MyDrive/BREAST_CANCER_FL'


### 1. Import Libraries

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import os
from PIL import Image
from tqdm import tqdm
import json
import pickle
import copy

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

# Import model architectures
import sys
sys.path.append('models')
from model_architectures import Hospital1_MLP, Hospital2_MLP, Hospital3_CNN

# Torchvision for image transforms
from torchvision import transforms

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

EMBEDDING_DIM =64

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Create results directories
os.makedirs('results/federated_learning', exist_ok=True)
os.makedirs('models/federated', exist_ok=True)

print("\n✓ All libraries imported successfully!")

Using device: cuda

✓ All libraries imported successfully!


### 2. Define Federated Learning Utility Functions

In [4]:
# FEDERATED LEARNING UTILITY FUNCTIONS

# FUNCTION 1 — get_shared_head_weights
# Extracts shared head weights from any hospital model
# These are the ONLY weights exchanged in federation
# Works identically for all three hospitals because
# SharedClassificationHead is identical across all
def get_shared_head_weights(model):
    """
    Extract shared head weights from a hospital model.

    Works for Hospital1_MLP, Hospital2_MLP, Hospital3_CNN
    because all three use the same SharedClassificationHead.

    Args:
        model : any hospital model

    Returns:
        state_dict of shared_head — ready for FedAvg
    """
    return {
        k: v.clone()
        for k, v in model.shared_head.state_dict().items()
    }


# FUNCTION 2 — set_shared_head_weights
# Loads globally averaged shared head weights back
# into each hospital model after FedAvg
def set_shared_head_weights(model, weights):
    """
    Load global shared head weights into a hospital model.

    Args:
        model   : any hospital model
        weights : globally averaged state_dict from FedAvg
    """
    model.shared_head.load_state_dict(weights)


# FUNCTION 3 — federated_averaging
# Performs weighted FedAvg on shared head weights
# Mathematically valid because all shared heads are
# identical in architecture and input dimension (64-dim)
def federated_averaging(weights_list, dataset_sizes):
    """
    Weighted federated averaging of shared head weights.

    FedAvg formula:
        global_w = sum(n_i / N * w_i) for all hospitals i
        where n_i = hospital i dataset size
              N   = total samples across all hospitals

    Args:
        weights_list  : list of shared_head state_dicts
                        one per hospital
        dataset_sizes : list of training sample counts
                        [570, 92, 1664] for H1, H2, H3

    Returns:
        avg_weights : globally averaged state_dict
    """
    total_size = sum(dataset_sizes)
    avg_weights = {}

    for key in weights_list[0].keys():
        # Weighted sum across all hospitals
        avg_weights[key] = sum(
            weights_list[i][key].float() *
            (dataset_sizes[i] / total_size)
            for i in range(len(weights_list))
        )

    return avg_weights


# FUNCTION 4 — aggregate_prototypes
# FedProto aggregation — averages class prototypes
# instead of weights
# Alternative to FedAvg for heterogeneous FL
def aggregate_prototypes(prototypes_list, dataset_sizes):
    """
    Weighted aggregation of class prototypes for FedProto.

    Each hospital contributes prototypes weighted by
    their dataset size — same weighting as FedAvg.

    Args:
        prototypes_list : list of dicts
                          each dict = {class_id: prototype_tensor}
                          one dict per hospital
        dataset_sizes   : list of training sample counts

    Returns:
        global_prototypes : dict {class_id: global_prototype}
    """
    total_size    = sum(dataset_sizes)
    num_classes   = len(prototypes_list[0])
    global_protos = {}

    for cls in range(num_classes):
        # Weighted average of class prototypes
        global_protos[cls] = sum(
            prototypes_list[i][cls].float() *
            (dataset_sizes[i] / total_size)
            for i in range(len(prototypes_list))
        )

    return global_protos


# FUNCTION 5 — train_local_epoch
# Trains one hospital model for one epoch locally
# Supports dual optimizer for encoder + shared head
def train_local_epoch(
    model,
    dataloader,
    criterion,
    optimizer_encoder,
    optimizer_head,
    device
):
    """
    Train hospital model for one local epoch.

    Uses dual optimizers:
        optimizer_encoder : updates private encoder
        optimizer_head    : updates shared head

    For Hospital 3 pass optimizer_projection as
    optimizer_encoder since backbone is frozen.

    Args:
        model            : any hospital model
        dataloader       : training DataLoader
        criterion        : BCEWithLogitsLoss
        optimizer_encoder: Adam for encoder/projection
        optimizer_head   : Adam for shared head
        device           : cpu or cuda

    Returns:
        epoch_loss : average loss
        epoch_acc  : accuracy
    """
    model.train()
    running_loss = 0.0
    all_preds    = []
    all_labels   = []

    for inputs, labels in dataloader:
        inputs = inputs.to(device).float()
        labels = labels.to(device).float().view(-1, 1)

        # Zero both optimizers
        optimizer_encoder.zero_grad(set_to_none=True)
        optimizer_head.zero_grad(set_to_none=True)

        # Forward pass
        outputs = model(inputs)
        loss    = criterion(outputs, labels)

        # Backward pass
        loss.backward()

        # Step both optimizers
        optimizer_encoder.step()
        optimizer_head.step()

        # Track metrics
        # Threshold at 0.0 for raw logits
        # logit > 0.0 equivalent to sigmoid(logit) > 0.5
        running_loss += loss.item() * inputs.size(0)
        preds = (outputs.detach() > 0.0).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc  = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc


# FUNCTION 6 — evaluate_federated_model
# Evaluates any hospital model on its test set
# Returns full metrics dict for comparison table
def evaluate_federated_model(model, dataloader, criterion, device):
    """
    Evaluate hospital model on test set.

    Args:
        model      : any hospital model
        dataloader : test DataLoader
        criterion  : BCEWithLogitsLoss
        device     : cpu or cuda

    Returns:
        metrics dict with loss, accuracy, precision,
        recall, f1, auc_roc
    """
    model.eval()
    running_loss = 0.0
    all_preds    = []
    all_probs    = []
    all_labels   = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device).float()
            labels = labels.to(device).float().view(-1, 1)

            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)

            # Threshold at 0.0 for raw logits
            preds = (outputs > 0.0).float()

            # Apply sigmoid for AUC computation
            probs = torch.sigmoid(outputs)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Flatten for sklearn
    all_labels = np.array(all_labels).flatten()
    all_preds  = np.array(all_preds).flatten()
    all_probs  = np.array(all_probs).flatten()

    epoch_loss = running_loss / len(dataloader.dataset)

    try:
        auc_roc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc_roc = 0.0

    metrics = {
        'loss'     : epoch_loss,
        'accuracy' : accuracy_score(all_labels, all_preds),
        'precision': precision_score(
            all_labels, all_preds, zero_division=0
        ),
        'recall'   : recall_score(
            all_labels, all_preds, zero_division=0
        ),
        'f1'       : f1_score(
            all_labels, all_preds, zero_division=0
        ),
        'auc_roc'  : auc_roc
    }

    return metrics


# FUNCTION 7 — evaluate_all_hospitals
# Evaluates all three hospitals after each FL round
# Returns combined results for comparison
def evaluate_all_hospitals(
    models,
    dataloaders,
    criterion,
    device,
    hospital_names=None
):
    """
    Evaluate all hospital models and return combined results.

    Args:
        models        : list [hospital1_model, hospital2_model,
                              hospital3_model]
        dataloaders   : list [test_loader_wdbc,
                              test_loader_coimbra,
                              test_loader_breakhis]
        criterion     : BCEWithLogitsLoss
        device        : cpu or cuda
        hospital_names: optional list of name strings

    Returns:
        results : dict {hospital_name: metrics_dict}
    """
    if hospital_names is None:
        hospital_names = [
            'Hospital 1 (WDBC)',
            'Hospital 2 (Coimbra)',
            'Hospital 3 (BreakHis)'
        ]

    results = {}
    for model, loader, name in zip(
        models, dataloaders, hospital_names
    ):
        metrics = evaluate_federated_model(
            model, loader, criterion, device
        )
        results[name] = metrics

    return results


# FUNCTION 8 — print_round_summary
# Prints a clean per-round summary during FL training
def print_round_summary(round_num, results, algorithm='FedAvg'):
    """
    Print clean summary after each federated round.

    Args:
        round_num : current FL round number
        results   : dict from evaluate_all_hospitals()
        algorithm : 'FedAvg' or 'FedProto'
    """
    print(f"\n  Round {round_num:02d} [{algorithm}] Results:")
    print(f"  {'Hospital':<25} {'Acc':>6} {'F1':>6} "
          f"{'Recall':>8} {'AUC':>6}")
    print(f"  {'─'*55}")
    for name, metrics in results.items():
        print(f"  {name:<25} "
              f"{metrics['accuracy']:>6.4f} "
              f"{metrics['f1']:>6.4f} "
              f"{metrics['recall']:>8.4f} "
              f"{metrics['auc_roc']:>6.4f}")


# Confirming all utility functions defined
print(" Federated learning utility functions defined:")
print("   - get_shared_head_weights()  : extract shared head")
print("   - set_shared_head_weights()  : load global weights")
print("   - federated_averaging()      : FedAvg aggregation")
print("   - aggregate_prototypes()     : FedProto aggregation")
print("   - train_local_epoch()        : local training step")
print("   - evaluate_federated_model() : evaluate one hospital")
print("   - evaluate_all_hospitals()   : evaluate all hospitals")
print("   - print_round_summary()      : per-round display")

 Federated learning utility functions defined:
   - get_shared_head_weights()  : extract shared head
   - set_shared_head_weights()  : load global weights
   - federated_averaging()      : FedAvg aggregation
   - aggregate_prototypes()     : FedProto aggregation
   - train_local_epoch()        : local training step
   - evaluate_federated_model() : evaluate one hospital
   - evaluate_all_hospitals()   : evaluate all hospitals
   - print_round_summary()      : per-round display


### 3. Load All Datasets

In [5]:
print("="*70)
print("LOADING ALL HOSPITAL DATASETS")
print("="*70)

# Hospital 1 - WDBC
print("\n Loading Hospital 1 (WDBC)...")

X_train_h1 = pd.read_csv('data/processed/wdbc/X_train.csv').values
y_train_h1 = pd.read_csv('data/processed/wdbc/y_train.csv').values.flatten()
X_test_h1 = pd.read_csv('data/processed/wdbc/X_test.csv').values
y_test_h1 = pd.read_csv('data/processed/wdbc/y_test.csv').values.flatten()

train_dataset_h1 = TensorDataset(
    torch.FloatTensor(X_train_h1),
    torch.FloatTensor(y_train_h1)
)
test_dataset_h1 = TensorDataset(
    torch.FloatTensor(X_test_h1),
    torch.FloatTensor(y_test_h1)
)

train_loader_h1 = DataLoader(train_dataset_h1, batch_size=32, shuffle=True)
test_loader_h1 = DataLoader(test_dataset_h1, batch_size=32, shuffle=False)

print(f"   Training samples: {len(train_dataset_h1)}")
print(f"   Test samples: {len(test_dataset_h1)}")

# Hospital 2 - Coimbra
print("\n Loading Hospital 2 (Coimbra)...")

X_train_h2 = pd.read_csv('data/processed/coimbra/X_train.csv').values
y_train_h2 = pd.read_csv('data/processed/coimbra/y_train.csv').values.flatten()
X_test_h2 = pd.read_csv('data/processed/coimbra/X_test.csv').values
y_test_h2 = pd.read_csv('data/processed/coimbra/y_test.csv').values.flatten()

train_dataset_h2 = TensorDataset(
    torch.FloatTensor(X_train_h2),
    torch.FloatTensor(y_train_h2)
)
test_dataset_h2 = TensorDataset(
    torch.FloatTensor(X_test_h2),
    torch.FloatTensor(y_test_h2)
)

train_loader_h2 = DataLoader(train_dataset_h2, batch_size=16, shuffle=True)
test_loader_h2 = DataLoader(test_dataset_h2, batch_size=16, shuffle=False)

print(f"   Training samples: {len(train_dataset_h2)}")
print(f"   Test samples: {len(test_dataset_h2)}")

# Hospital 3 - BreakHis
print("\n Loading Hospital 3 (BreakHis)...")

# Define BreakHis Dataset class
class BreakHisDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []

        # LoadING benign images
        benign_dir = os.path.join(root_dir, 'benign')
        if os.path.exists(benign_dir):
            for img_name in os.listdir(benign_dir):
                if img_name.endswith('.png'):
                    self.images.append(os.path.join(benign_dir, img_name))
                    self.labels.append(0)

        # LoadING malignant images
        malignant_dir = os.path.join(root_dir, 'malignant')
        if os.path.exists(malignant_dir):
            for img_name in os.listdir(malignant_dir):
                if img_name.endswith('.png'):
                    self.images.append(os.path.join(malignant_dir, img_name))
                    self.labels.append(1)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# Transforms
train_transform_h3 = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform_h3 = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset_h3 = BreakHisDataset('data/processed/breakhis/train', transform=train_transform_h3)
test_dataset_h3 = BreakHisDataset('data/processed/breakhis/test', transform=test_transform_h3)

train_loader_h3 = DataLoader(train_dataset_h3, batch_size=32, shuffle=True, num_workers=2)
test_loader_h3 = DataLoader(test_dataset_h3, batch_size=32, shuffle=False, num_workers=2)

print(f"   Training samples: {len(train_dataset_h3)}")
print(f"   Test samples: {len(test_dataset_h3)}")

# Dataset sizes for weighted averaging
dataset_sizes = [
    len(train_dataset_h1),
    len(train_dataset_h2),
    len(train_dataset_h3)
]

print("\n" + "="*70)
print("DATASET SIZES FOR WEIGHTED AVERAGING")
print("="*70)
print(f"Hospital 1 (WDBC):     {dataset_sizes[0]:>6} samples ({dataset_sizes[0]/sum(dataset_sizes)*100:.1f}%)")
print(f"Hospital 2 (Coimbra):  {dataset_sizes[1]:>6} samples ({dataset_sizes[1]/sum(dataset_sizes)*100:.1f}%)")
print(f"Hospital 3 (BreakHis): {dataset_sizes[2]:>6} samples ({dataset_sizes[2]/sum(dataset_sizes)*100:.1f}%)")
print(f"Total:                 {sum(dataset_sizes):>6} samples")
print("="*70)

LOADING ALL HOSPITAL DATASETS

 Loading Hospital 1 (WDBC)...
   Training samples: 570
   Test samples: 114

 Loading Hospital 2 (Coimbra)...
   Training samples: 92
   Test samples: 24

 Loading Hospital 3 (BreakHis)...
   Training samples: 1990
   Test samples: 743

DATASET SIZES FOR WEIGHTED AVERAGING
Hospital 1 (WDBC):        570 samples (21.5%)
Hospital 2 (Coimbra):      92 samples (3.5%)
Hospital 3 (BreakHis):   1990 samples (75.0%)
Total:                   2652 samples


### 4. Initialize Models for Federated Learning

In [6]:
os.makedirs('results/federated', exist_ok=True)
os.makedirs('models/federated', exist_ok=True)

print("\n" + "="*70)
print("  PHASE 4 — INITIALIZING FEDERATED LEARNING MODELS")
print("="*70)

# STEP 1 — Initializing models with new architecture
# Loading locally trained weights from Phase 3 as starting point
print("\n Loading Phase 3 trained weights...")

# Hospital 1
model_h1 = Hospital1_MLP(
    input_size=23,
    dropout_rate=0.3
).to(device)
model_h1.load_state_dict(
    torch.load(
        'models/trained/hospital1_local.pth',
        map_location=device
    )
)
print("    Hospital 1 weights loaded from Phase 3")

# Hospital 2
model_h2 = Hospital2_MLP(
    input_size=9,
    dropout_rate=0.3
).to(device)
model_h2.load_state_dict(
    torch.load(
        'models/trained/hospital2_local.pth',
        map_location=device
    )
)
print("    Hospital 2 weights loaded from Phase 3")

# Hospital 3
model_h3 = Hospital3_CNN(dropout_rate=0.2).to(device)
model_h3.load_state_dict(
    torch.load(
        'models/trained/hospital3_local.pth',
        map_location=device
    )
)
print("    Hospital 3 weights loaded from Phase 3")

# STEP 2 — Verifying shared head compatibility
# Critical check — confirms FedAvg is mathematically valid
print("\n Verifying shared head compatibility...")

dummy_h1 = torch.randn(2, 23).to(device)
dummy_h2 = torch.randn(2, 9).to(device)
dummy_h3 = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    model_h1.eval()
    model_h2.eval()
    model_h3.eval()

    emb_h1 = model_h1.get_embedding(dummy_h1)
    emb_h2 = model_h2.get_embedding(dummy_h2)
    emb_h3 = model_h3.get_embedding(dummy_h3)

    # Feeding each hospital's embedding through all shared heads
    h1_head_accepts_h2 = model_h1.shared_head(emb_h2).shape[-1] == 1
    h1_head_accepts_h3 = model_h1.shared_head(emb_h3).shape[-1] == 1

assert emb_h1.shape[1] == EMBEDDING_DIM, "H1 embedding dim mismatch"
assert emb_h2.shape[1] == EMBEDDING_DIM, "H2 embedding dim mismatch"
assert emb_h3.shape[1] == EMBEDDING_DIM, "H3 embedding dim mismatch"
assert h1_head_accepts_h2, "H1 head rejects H2 embedding"
assert h1_head_accepts_h3, "H1 head rejects H3 embedding"

print(f"    H1 embedding shape : {emb_h1.shape}")
print(f"    H2 embedding shape : {emb_h2.shape}")
print(f"    H3 embedding shape : {emb_h3.shape}")
print(f"    All embeddings     : {EMBEDDING_DIM}-dim (compatible)")
print(f"    FedAvg             : mathematically valid ")

model_h1.train()
model_h2.train()
model_h3.train()

# STEP 3 — Dual optimizers per hospital
# Encoder gets higher LR — learning local patterns
# Shared head gets lower LR — conservative before FedAvg

# Hospital 1 optimizers
optimizer_h1_encoder = optim.Adam(
    model_h1.encoder.parameters(),
    lr=0.001
)
optimizer_h1_head = optim.Adam(
    model_h1.shared_head.parameters(),
    lr=0.0005
)

# Hospital 2 optimizers
optimizer_h2_encoder = optim.Adam(
    model_h2.encoder.parameters(),
    lr=0.001
)
optimizer_h2_head = optim.Adam(
    model_h2.shared_head.parameters(),
    lr=0.0005
)

# Hospital 3 optimizers
# Uses projection optimizer instead of encoder
# Backbone is frozen — no optimizer needed
optimizer_h3_projection = optim.Adam(
    model_h3.encoder.projection.parameters(),
    lr=0.0001
)
optimizer_h3_head = optim.Adam(
    model_h3.shared_head.parameters(),
    lr=0.00005
)

# STEP 4 — Schedulers per optimizer
scheduler_h1_encoder = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_h1_encoder, mode='min', patience=5, factor=0.5
)
scheduler_h1_head = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_h1_head, mode='min', patience=5, factor=0.5
)
scheduler_h2_encoder = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_h2_encoder, mode='min', patience=3, factor=0.5
)
scheduler_h2_head = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_h2_head, mode='min', patience=3, factor=0.5
)
scheduler_h3_projection = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_h3_projection, mode='min', patience=5, factor=0.5
)
scheduler_h3_head = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_h3_head, mode='min', patience=5, factor=0.5
)

# STEP 5 — Loss function
criterion = nn.BCEWithLogitsLoss()

# STEP 6 — Printing configuration summary
h1_encoder_params = sum(
    p.numel() for p in model_h1.encoder.parameters()
)
h1_head_params = sum(
    p.numel() for p in model_h1.shared_head.parameters()
)
h2_encoder_params = sum(
    p.numel() for p in model_h2.encoder.parameters()
)
h2_head_params = sum(
    p.numel() for p in model_h2.shared_head.parameters()
)
h3_backbone_params = sum(
    p.numel() for p in model_h3.encoder.backbone.parameters()
)
h3_projection_params = sum(
    p.numel() for p in model_h3.encoder.projection.parameters()
)
h3_head_params = sum(
    p.numel() for p in model_h3.shared_head.parameters()
)

print(f"\n{'='*55}")
print(f"  FL MODEL INITIALIZATION SUMMARY")
print(f"{'='*55}")
print(f"  {'Component':<30} {'Params':>10} {'Role'}")
print(f"  {'─'*55}")
print(f"  {'H1 Encoder':<30} {h1_encoder_params:>10,}   private")
print(f"  {'H1 Shared Head':<30} {h1_head_params:>10,}   federated")
print(f"  {'H2 Encoder':<30} {h2_encoder_params:>10,}   private")
print(f"  {'H2 Shared Head':<30} {h2_head_params:>10,}   federated")
print(f"  {'H3 Backbone (frozen)':<30} {h3_backbone_params:>10,}   frozen")
print(f"  {'H3 Projection':<30} {h3_projection_params:>10,}   private")
print(f"  {'H3 Shared Head':<30} {h3_head_params:>10,}   federated")
print(f"  {'─'*55}")
print(f"  {'FL params per round':<30} {h1_head_params:>10,}   (all heads identical)")
print(f"  {'Embedding dimension':<30} {EMBEDDING_DIM:>10}   (shared space)")
print(f"  {'─'*55}")
print(f"  Loss function  : BCEWithLogitsLoss")
print(f"  H1 encoder LR  : 0.001")
print(f"  H1 head LR     : 0.0005")
print(f"  H2 encoder LR  : 0.001")
print(f"  H2 head LR     : 0.0005")
print(f"  H3 proj LR     : 0.0001")
print(f"  H3 head LR     : 0.00005")
print(f"  Weights source : Phase 3 trained models")
print(f"{'='*55}")


  PHASE 4 — INITIALIZING FEDERATED LEARNING MODELS

 Loading Phase 3 trained weights...
    Hospital 1 weights loaded from Phase 3
    Hospital 2 weights loaded from Phase 3
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 54.8MB/s]


    Hospital 3 weights loaded from Phase 3

 Verifying shared head compatibility...
    H1 embedding shape : torch.Size([2, 64])
    H2 embedding shape : torch.Size([2, 64])
    H3 embedding shape : torch.Size([2, 64])
    All embeddings     : 64-dim (compatible)
    FedAvg             : mathematically valid 

  FL MODEL INITIALIZATION SUMMARY
  Component                          Params Role
  ───────────────────────────────────────────────────────
  H1 Encoder                         11,712   private
  H1 Shared Head                      2,177   federated
  H2 Encoder                          2,624   private
  H2 Shared Head                      2,177   federated
  H3 Backbone (frozen)            2,223,872   frozen
  H3 Projection                     345,024   private
  H3 Shared Head                      2,177   federated
  ───────────────────────────────────────────────────────
  FL params per round                 2,177   (all heads identical)
  Embedding dimension                 

### 5. Federated Learning Training Loop

In [7]:
print("\n" + "="*70)
print("  PHASE 4 — FEDERATED LEARNING TRAINING LOOP (FedAvg)")
print("="*70)

# FL Configuration
num_fl_rounds = 25 # 25 is a standard starting point in FL literature for small to medium scale experiments.
local_epochs  = 2 # 2 epochs keeps local updates meaningful but small enough that FedAvg still works effectively
best_fl_loss  = float('inf')

fl_history = {
    'round'       : [],
    'h1_train_loss': [], 'h1_train_acc': [],
    'h2_train_loss': [], 'h2_train_acc': [],
    'h3_train_loss': [], 'h3_train_acc': [],
    'h1_test_acc' : [], 'h1_test_f1': [], 'h1_test_auc': [],
    'h2_test_acc' : [], 'h2_test_f1': [], 'h2_test_auc': [],
    'h3_test_acc' : [], 'h3_test_f1': [], 'h3_test_auc': [],
    'avg_test_acc': [], 'avg_test_f1': [], 'avg_test_auc': []
}

print(f"\n  FL Configuration:")
print(f"  Algorithm         : Weighted FedAvg")
print(f"  Total FL rounds   : {num_fl_rounds}")
print(f"  Local epochs/round: {local_epochs}")
print(f"  Hospitals         : 3")
print(f"  Embedding dim     : {EMBEDDING_DIM}")
print(f"  FL params/round   : {h1_head_params:,} per hospital")
print(f"  Dataset weights   :")
print(f"    H1 : {dataset_sizes[0]:,} / {sum(dataset_sizes):,} "
      f"= {dataset_sizes[0]/sum(dataset_sizes)*100:.1f}%")
print(f"    H2 : {dataset_sizes[1]:,} / {sum(dataset_sizes):,} "
      f"= {dataset_sizes[1]/sum(dataset_sizes)*100:.1f}%")
print(f"    H3 : {dataset_sizes[2]:,} / {sum(dataset_sizes):,} "
      f"= {dataset_sizes[2]/sum(dataset_sizes)*100:.1f}%")
print(f"\n  Starting FL rounds...\n")

# MAIN FEDERATED LEARNING LOOP
for fl_round in range(num_fl_rounds):

    print(f"\n{'='*55}")
    print(f"  FL ROUND {fl_round+1:02d}/{num_fl_rounds} — FedAvg")
    print(f"{'='*55}")

    # STEP 1 — LOCAL TRAINING AT EACH HOSPITAL
    # Each hospital trains for local_epochs on its own data
    # Both encoder and shared head updated locally
    print(f"\n   Local Training ({local_epochs} epochs each)...")

    # Hospital 1 local training
    h1_losses, h1_accs = [], []
    for epoch in range(local_epochs):
        loss, acc = train_local_epoch(
            model_h1,
            train_loader_h1,
            criterion,
            optimizer_h1_encoder,   # encoder optimizer
            optimizer_h1_head,      # shared head optimizer
            device
        )
        h1_losses.append(loss)
        h1_accs.append(acc)
    avg_h1_loss = np.mean(h1_losses)
    avg_h1_acc  = np.mean(h1_accs)
    print(f"     H1 (WDBC)    : "
          f"Loss={avg_h1_loss:.4f} Acc={avg_h1_acc:.4f}")

    # Hospital 2 local training
    h2_losses, h2_accs = [], []
    for epoch in range(local_epochs):
        loss, acc = train_local_epoch(
            model_h2,
            train_loader_h2,
            criterion,
            optimizer_h2_encoder,
            optimizer_h2_head,
            device
        )
        h2_losses.append(loss)
        h2_accs.append(acc)
    avg_h2_loss = np.mean(h2_losses)
    avg_h2_acc  = np.mean(h2_accs)
    print(f"     H2 (Coimbra) : "
          f"Loss={avg_h2_loss:.4f} Acc={avg_h2_acc:.4f}")

    # Hospital 3 local training
    # Uses projection optimizer instead of encoder optimizer
    h3_losses, h3_accs = [], []
    for epoch in range(local_epochs):
        loss, acc = train_local_epoch(
            model_h3,
            train_loader_h3,
            criterion,
            optimizer_h3_projection,  # projection instead of encoder
            optimizer_h3_head,
            device
        )
        h3_losses.append(loss)
        h3_accs.append(acc)
    avg_h3_loss = np.mean(h3_losses)
    avg_h3_acc  = np.mean(h3_accs)
    print(f"     H3 (BreakHis): "
          f"Loss={avg_h3_loss:.4f} Acc={avg_h3_acc:.4f}")

    # STEP 2 — EXTRACT SHARED HEAD WEIGHTS
    # Only shared head weights leave each hospital
    # Encoder weights stay private always
    print(f"\n   Extracting shared head weights...")

    weights_h1 = get_shared_head_weights(model_h1)
    weights_h2 = get_shared_head_weights(model_h2)
    weights_h3 = get_shared_head_weights(model_h3)

    print(f"      H1 shared head : "
          f"{sum(v.numel() for v in weights_h1.values()):,} params")
    print(f"      H2 shared head : "
          f"{sum(v.numel() for v in weights_h2.values()):,} params")
    print(f"      H3 shared head : "
          f"{sum(v.numel() for v in weights_h3.values()):,} params")

    # STEP 3 — FEDERATED AVERAGING (SERVER SIDE)
    # Central server computes weighted average of
    # shared head weights from all three hospitals
    # Mathematically valid because all shared heads are
    # identical architecture with 64-dim input
    print(f"\n   Central server — FedAvg...")

    global_weights = federated_averaging(
        weights_list  = [weights_h1, weights_h2, weights_h3],
        dataset_sizes = dataset_sizes
    )

    print(f"      Weighted average computed")
    print(f"      H1 weight : "
          f"{dataset_sizes[0]/sum(dataset_sizes)*100:.1f}%")
    print(f"      H2 weight : "
          f"{dataset_sizes[1]/sum(dataset_sizes)*100:.1f}%")
    print(f"      H3 weight : "
          f"{dataset_sizes[2]/sum(dataset_sizes)*100:.1f}%")

    # STEP 4 — DISTRIBUTING GLOBAL WEIGHTS BACK
    # All hospitals receive the same global shared head
    # Each hospital loads global weights into their shared head
    # Encoder weights remain untouched — still private
    print(f"\n   Distributing global weights to all hospitals...")

    set_shared_head_weights(model_h1, global_weights)
    set_shared_head_weights(model_h2, global_weights)
    set_shared_head_weights(model_h3, global_weights)

    print(f"      Global shared head loaded into H1")
    print(f"      Global shared head loaded into H2")
    print(f"      Global shared head loaded into H3")

    # STEP 5 — STEP SCHEDULERS
    # Use average validation loss across all hospitals
    avg_train_loss = np.mean([avg_h1_loss, avg_h2_loss, avg_h3_loss])

    scheduler_h1_encoder.step(avg_h1_loss)
    scheduler_h1_head.step(avg_h1_loss)
    scheduler_h2_encoder.step(avg_h2_loss)
    scheduler_h2_head.step(avg_h2_loss)
    scheduler_h3_projection.step(avg_h3_loss)
    scheduler_h3_head.step(avg_h3_loss)

    # STEP 6 — EVALUATING ALL HOSPITALS
    print(f"\n   Evaluating all hospitals...")

    metrics_h1 = evaluate_federated_model(
        model_h1, test_loader_h1, criterion, device
    )
    metrics_h2 = evaluate_federated_model(
        model_h2, test_loader_h2, criterion, device
    )
    metrics_h3 = evaluate_federated_model(
        model_h3, test_loader_h3, criterion, device
    )

    # Weighted average metrics across hospitals
    total = sum(dataset_sizes)
    avg_acc = (
        dataset_sizes[0]/total * metrics_h1['accuracy'] +
        dataset_sizes[1]/total * metrics_h2['accuracy'] +
        dataset_sizes[2]/total * metrics_h3['accuracy']
    )
    avg_f1 = (
        dataset_sizes[0]/total * metrics_h1['f1'] +
        dataset_sizes[1]/total * metrics_h2['f1'] +
        dataset_sizes[2]/total * metrics_h3['f1']
    )
    avg_auc = (
        dataset_sizes[0]/total * metrics_h1['auc_roc'] +
        dataset_sizes[1]/total * metrics_h2['auc_roc'] +
        dataset_sizes[2]/total * metrics_h3['auc_roc']
    )

    print_round_summary(
        fl_round + 1,
        {
            'Hospital 1 (WDBC)'    : metrics_h1,
            'Hospital 2 (Coimbra)' : metrics_h2,
            'Hospital 3 (BreakHis)': metrics_h3
        },
        algorithm='FedAvg'
    )
    print(f"\n  Weighted avg — "
          f"Acc={avg_acc:.4f} "
          f"F1={avg_f1:.4f} "
          f"AUC={avg_auc:.4f}")

    # STEP 7 — SAVING HISTORY
    fl_history['round'].append(fl_round + 1)
    fl_history['h1_train_loss'].append(avg_h1_loss)
    fl_history['h1_train_acc'].append(avg_h1_acc)
    fl_history['h2_train_loss'].append(avg_h2_loss)
    fl_history['h2_train_acc'].append(avg_h2_acc)
    fl_history['h3_train_loss'].append(avg_h3_loss)
    fl_history['h3_train_acc'].append(avg_h3_acc)
    fl_history['h1_test_acc'].append(metrics_h1['accuracy'])
    fl_history['h1_test_f1'].append(metrics_h1['f1'])
    fl_history['h1_test_auc'].append(metrics_h1['auc_roc'])
    fl_history['h2_test_acc'].append(metrics_h2['accuracy'])
    fl_history['h2_test_f1'].append(metrics_h2['f1'])
    fl_history['h2_test_auc'].append(metrics_h2['auc_roc'])
    fl_history['h3_test_acc'].append(metrics_h3['accuracy'])
    fl_history['h3_test_f1'].append(metrics_h3['f1'])
    fl_history['h3_test_auc'].append(metrics_h3['auc_roc'])
    fl_history['avg_test_acc'].append(avg_acc)
    fl_history['avg_test_f1'].append(avg_f1)
    fl_history['avg_test_auc'].append(avg_auc)

    # STEP 8 — SAVING BEST GLOBAL MODEL
    current_avg_loss = np.mean([
        metrics_h1['loss'],
        metrics_h2['loss'],
        metrics_h3['loss']
    ])
    if current_avg_loss < best_fl_loss:
        best_fl_loss = current_avg_loss
        torch.save(
            global_weights,
            'models/federated/best_global_shared_head.pth'
        )
        torch.save(
            model_h1.state_dict(),
            'models/federated/best_h1_fedavg.pth'
        )
        torch.save(
            model_h2.state_dict(),
            'models/federated/best_h2_fedavg.pth'
        )
        torch.save(
            model_h3.state_dict(),
            'models/federated/best_h3_fedavg.pth'
        )
        print(f"\n   New best model saved "
              f"(avg loss: {best_fl_loss:.4f})")

print("\n" + "="*70)
print("  FEDERATED LEARNING COMPLETE!")
print("="*70)
print(f"  Total rounds completed : {num_fl_rounds}")
print(f"  Best average test loss : {best_fl_loss:.4f}")
print(f"  Saved files:")
print(f"    models/federated/best_global_shared_head.pth")
print(f"    models/federated/best_h1_fedavg.pth")
print(f"    models/federated/best_h2_fedavg.pth")
print(f"    models/federated/best_h3_fedavg.pth")
print("="*70)


  PHASE 4 — FEDERATED LEARNING TRAINING LOOP (FedAvg)

  FL Configuration:
  Algorithm         : Weighted FedAvg
  Total FL rounds   : 25
  Local epochs/round: 2
  Hospitals         : 3
  Embedding dim     : 64
  FL params/round   : 2,177 per hospital
  Dataset weights   :
    H1 : 570 / 2,652 = 21.5%
    H2 : 92 / 2,652 = 3.5%
    H3 : 1,990 / 2,652 = 75.0%

  Starting FL rounds...


  FL ROUND 01/25 — FedAvg

   Local Training (2 epochs each)...
     H1 (WDBC)    : Loss=0.0631 Acc=0.9816
     H2 (Coimbra) : Loss=0.4469 Acc=0.8152
     H3 (BreakHis): Loss=0.1894 Acc=0.9719

   Extracting shared head weights...
      H1 shared head : 2,242 params
      H2 shared head : 2,242 params
      H3 shared head : 2,242 params

   Central server — FedAvg...
      Weighted average computed
      H1 weight : 21.5%
      H2 weight : 3.5%
      H3 weight : 75.0%

   Distributing global weights to all hospitals...
      Global shared head loaded into H1
      Global shared head loaded into H2
      

### 6. Save Federated Models

In [8]:
os.makedirs('models/federated', exist_ok=True)
os.makedirs('results/federated', exist_ok=True)

print("\n" + "="*70)
print("  PHASE 4 — SAVING FEDERATED MODELS")
print("="*70)

# STEP 1 — Saving full federated models
print("\n Saving full federated models...")

torch.save(
    model_h1.state_dict(),
    'models/federated/hospital1_federated.pth'
)
torch.save(
    model_h2.state_dict(),
    'models/federated/hospital2_federated.pth'
)
torch.save(
    model_h3.state_dict(),
    'models/federated/hospital3_federated.pth'
)
print("    hospital1_federated.pth")
print("    hospital2_federated.pth")
print("    hospital3_federated.pth")

# STEP 2 — Saving encoders separately (private components)
# These never left their hospitals during FL
# Saved for completeness and future inference
print("\n Saving private encoders...")

torch.save(
    model_h1.encoder.state_dict(),
    'models/federated/hospital1_encoder_federated.pth'
)
torch.save(
    model_h2.encoder.state_dict(),
    'models/federated/hospital2_encoder_federated.pth'
)
torch.save(
    model_h3.encoder.state_dict(),
    'models/federated/hospital3_encoder_federated.pth'
)
print("    hospital1_encoder_federated.pth")
print("    hospital2_encoder_federated.pth")
print("    hospital3_encoder_federated.pth")

# STEP 3 — Saving final global shared head
# This is the most important FL artifact
# Represents the aggregated knowledge from all 3 hospitals
print("\n Saving final global shared head...")

final_global_weights = federated_averaging(
    weights_list  = [
        get_shared_head_weights(model_h1),
        get_shared_head_weights(model_h2),
        get_shared_head_weights(model_h3)
    ],
    dataset_sizes = dataset_sizes
)

torch.save(
    final_global_weights,
    'models/federated/final_global_shared_head.pth'
)
print("    final_global_shared_head.pth")

# Save shared head from each hospital separately
torch.save(
    model_h1.shared_head.state_dict(),
    'models/federated/hospital1_shared_head_federated.pth'
)
torch.save(
    model_h2.shared_head.state_dict(),
    'models/federated/hospital2_shared_head_federated.pth'
)
torch.save(
    model_h3.shared_head.state_dict(),
    'models/federated/hospital3_shared_head_federated.pth'
)
print("    hospital1_shared_head_federated.pth")
print("    hospital2_shared_head_federated.pth")
print("    hospital3_shared_head_federated.pth")

# STEP 4 — Saving FL training history
# Convert numpy values to Python floats for JSON
print("\n Saving FL training history...")

fl_history_serializable = {
    k: [float(v) for v in vals]
    if isinstance(vals, list) and len(vals) > 0
    and isinstance(vals[0], (np.floating, np.integer, float, int))
    else vals
    for k, vals in fl_history.items()
}

with open(
    'results/federated/fl_history.json',
    'w',
    encoding='utf-8'
) as f:
    json.dump(fl_history_serializable, f, indent=4)

print("    fl_history.json")

# STEP 5 — Verifying all files saved correctly
print("\n Verifying saved files...")

files_to_verify = [
    'models/federated/hospital1_federated.pth',
    'models/federated/hospital2_federated.pth',
    'models/federated/hospital3_federated.pth',
    'models/federated/hospital1_encoder_federated.pth',
    'models/federated/hospital2_encoder_federated.pth',
    'models/federated/hospital3_encoder_federated.pth',
    'models/federated/final_global_shared_head.pth',
    'models/federated/hospital1_shared_head_federated.pth',
    'models/federated/hospital2_shared_head_federated.pth',
    'models/federated/hospital3_shared_head_federated.pth',
    'results/federated/fl_history.json',
]

all_saved = True
for filepath in files_to_verify:
    if os.path.exists(filepath):
        size_kb = os.path.getsize(filepath) / 1024
        print(f"    {filepath:<55} ({size_kb:.1f} KB)")
    else:
        print(f"    {filepath} — NOT FOUND")
        all_saved = False

# STEP 6 — Final summary
h1_encoder_params    = sum(
    p.numel() for p in model_h1.encoder.parameters()
)
h1_head_params       = sum(
    p.numel() for p in model_h1.shared_head.parameters()
)
h2_encoder_params    = sum(
    p.numel() for p in model_h2.encoder.parameters()
)
h3_backbone_params   = sum(
    p.numel() for p in model_h3.encoder.backbone.parameters()
)
h3_projection_params = sum(
    p.numel() for p in model_h3.encoder.projection.parameters()
)

print(f"\n{'='*55}")
print(f"  PHASE 4 SAVING SUMMARY")
print(f"{'='*55}")
print(f"  FL algorithm       : FedAvg")
print(f"  FL rounds          : {num_fl_rounds}")
print(f"  Embedding dim      : {EMBEDDING_DIM}")
print(f"  {'─'*53}")
print(f"  Component          Params     Saved As")
print(f"  {'─'*53}")
print(f"  H1 encoder         {h1_encoder_params:>8,}   private (federated)")
print(f"  H2 encoder         {h2_encoder_params:>8,}   private (federated)")
print(f"  H3 backbone        {h3_backbone_params:>8,}   frozen  (federated)")
print(f"  H3 projection      {h3_projection_params:>8,}   private (federated)")
print(f"  Global shared head {h1_head_params:>8,}   FL artifact")
print(f"  {'─'*53}")
print(f"  Total files saved  : {len(files_to_verify)}")
print(f"  All files present  : {' Yes' if all_saved else ' No'}")
print(f"  {'─'*53}")
print(f"  Next Step: Phase 5 — Differential Privacy Analysis")
print(f"{'='*55}")


  PHASE 4 — SAVING FEDERATED MODELS

 Saving full federated models...
    hospital1_federated.pth
    hospital2_federated.pth
    hospital3_federated.pth

 Saving private encoders...
    hospital1_encoder_federated.pth
    hospital2_encoder_federated.pth
    hospital3_encoder_federated.pth

 Saving final global shared head...
    final_global_shared_head.pth
    hospital1_shared_head_federated.pth
    hospital2_shared_head_federated.pth
    hospital3_shared_head_federated.pth

 Saving FL training history...
    fl_history.json

 Verifying saved files...
    models/federated/hospital1_federated.pth                (64.4 KB)
    models/federated/hospital2_federated.pth                (28.2 KB)
    models/federated/hospital3_federated.pth                (10304.7 KB)
    models/federated/hospital1_encoder_federated.pth        (52.9 KB)
    models/federated/hospital2_encoder_federated.pth        (16.7 KB)
    models/federated/hospital3_encoder_federated.pth        (10291.2 KB)
    models/fe

### 7. Plot Federated Learning Results

In [9]:
os.makedirs('results/federated', exist_ok=True)

print("\n" + "="*70)
print("  PHASE 4 — GENERATING FEDERATED LEARNING PLOTS")
print("="*70)

# Color scheme — consistent across all plots
COLOR_H1  = '#2ECC71'   # green  — Hospital 1 WDBC
COLOR_H2  = '#E74C3C'   # red    — Hospital 2 Coimbra
COLOR_H3  = '#3498DB'   # blue   — Hospital 3 BreakHis
COLOR_AVG = '#9B59B6'   # purple — weighted average

rounds = fl_history['round']

# PLOT 1 — Testing Accuracy over FL Rounds
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    rounds, fl_history['h1_test_acc'],
    marker='o', label='Hospital 1 (WDBC)',
    linewidth=2, color=COLOR_H1, markersize=5
)
ax.plot(
    rounds, fl_history['h2_test_acc'],
    marker='s', label='Hospital 2 (Coimbra)',
    linewidth=2, color=COLOR_H2, markersize=5
)
ax.plot(
    rounds, fl_history['h3_test_acc'],
    marker='^', label='Hospital 3 (BreakHis)',
    linewidth=2, color=COLOR_H3, markersize=5
)
ax.plot(
    rounds, fl_history['avg_test_acc'],
    marker='D', label='Weighted Average',
    linewidth=2.5, color=COLOR_AVG,
    markersize=5, linestyle='--'
)

# Annotating best round for weighted average
best_round_acc = rounds[np.argmax(fl_history['avg_test_acc'])]
best_acc_val   = max(fl_history['avg_test_acc'])
ax.axvline(
    x=best_round_acc, color=COLOR_AVG,
    linestyle=':', alpha=0.6
)
ax.annotate(
    f'Best avg: {best_acc_val:.4f}\n(round {best_round_acc})',
    xy=(best_round_acc, best_acc_val),
    xytext=(best_round_acc + 1, best_acc_val - 0.05),
    fontsize=8, color=COLOR_AVG
)

ax.set_xlabel('FL Round', fontsize=12, fontweight='bold')
ax.set_ylabel('Test Accuracy', fontsize=12, fontweight='bold')
ax.set_title(
    'Federated Learning (FedAvg) — Test Accuracy Over Rounds\n'
    f'Encoder (private) + SharedHead (federated) | '
    f'Embedding dim = {EMBEDDING_DIM}',
    fontsize=12, fontweight='bold'
)
ax.set_ylim([0.0, 1.05])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    'results/federated/fl_test_accuracy.png',
    dpi=300, bbox_inches='tight'
)
plt.close()
print("    fl_test_accuracy.png")

# PLOT 2 — F1-Score over FL Rounds
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    rounds, fl_history['h1_test_f1'],
    marker='o', label='Hospital 1 (WDBC)',
    linewidth=2, color=COLOR_H1, markersize=5
)
ax.plot(
    rounds, fl_history['h2_test_f1'],
    marker='s', label='Hospital 2 (Coimbra)',
    linewidth=2, color=COLOR_H2, markersize=5
)
ax.plot(
    rounds, fl_history['h3_test_f1'],
    marker='^', label='Hospital 3 (BreakHis)',
    linewidth=2, color=COLOR_H3, markersize=5
)
ax.plot(
    rounds, fl_history['avg_test_f1'],
    marker='D', label='Weighted Average',
    linewidth=2.5, color=COLOR_AVG,
    markersize=5, linestyle='--'
)

best_round_f1 = rounds[np.argmax(fl_history['avg_test_f1'])]
best_f1_val   = max(fl_history['avg_test_f1'])
ax.axvline(
    x=best_round_f1, color=COLOR_AVG,
    linestyle=':', alpha=0.6
)
ax.annotate(
    f'Best avg: {best_f1_val:.4f}\n(round {best_round_f1})',
    xy=(best_round_f1, best_f1_val),
    xytext=(best_round_f1 + 1, best_f1_val - 0.05),
    fontsize=8, color=COLOR_AVG
)

ax.set_xlabel('FL Round', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title(
    'Federated Learning (FedAvg) — F1-Score Over Rounds\n'
    'Clinical metric — recall weighted equally with precision',
    fontsize=12, fontweight='bold'
)
ax.set_ylim([0.0, 1.05])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    'results/federated/fl_f1_score.png',
    dpi=300, bbox_inches='tight'
)
plt.close()
print("    fl_f1_score.png")

# PLOT 3 — AUC-ROC over FL Rounds
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    rounds, fl_history['h1_test_auc'],
    marker='o', label='Hospital 1 (WDBC)',
    linewidth=2, color=COLOR_H1, markersize=5
)
ax.plot(
    rounds, fl_history['h2_test_auc'],
    marker='s', label='Hospital 2 (Coimbra)',
    linewidth=2, color=COLOR_H2, markersize=5
)
ax.plot(
    rounds, fl_history['h3_test_auc'],
    marker='^', label='Hospital 3 (BreakHis)',
    linewidth=2, color=COLOR_H3, markersize=5
)
ax.plot(
    rounds, fl_history['avg_test_auc'],
    marker='D', label='Weighted Average',
    linewidth=2.5, color=COLOR_AVG,
    markersize=5, linestyle='--'
)

# Random classifier baseline
ax.axhline(
    y=0.5, color='gray',
    linestyle='--', alpha=0.5,
    label='Random classifier (0.5)'
)

best_round_auc = rounds[np.argmax(fl_history['avg_test_auc'])]
best_auc_val   = max(fl_history['avg_test_auc'])
ax.axvline(
    x=best_round_auc, color=COLOR_AVG,
    linestyle=':', alpha=0.6
)
ax.annotate(
    f'Best avg: {best_auc_val:.4f}\n(round {best_round_auc})',
    xy=(best_round_auc, best_auc_val),
    xytext=(best_round_auc + 1, best_auc_val - 0.05),
    fontsize=8, color=COLOR_AVG
)

ax.set_xlabel('FL Round', fontsize=12, fontweight='bold')
ax.set_ylabel('AUC-ROC', fontsize=12, fontweight='bold')
ax.set_title(
    'Federated Learning (FedAvg) — AUC-ROC Over Rounds\n'
    'Discriminative ability across all classification thresholds',
    fontsize=12, fontweight='bold'
)
ax.set_ylim([0.0, 1.05])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    'results/federated/fl_auc_roc.png',
    dpi=300, bbox_inches='tight'
)
plt.close()
print("    fl_auc_roc.png")

# PLOT 4 — Training Loss over FL Rounds (new)
# Shows local training loss per hospital per round
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    rounds, fl_history['h1_train_loss'],
    marker='o', label='Hospital 1 (WDBC)',
    linewidth=2, color=COLOR_H1,
    markersize=4, alpha=0.8
)
ax.plot(
    rounds, fl_history['h2_train_loss'],
    marker='s', label='Hospital 2 (Coimbra)',
    linewidth=2, color=COLOR_H2,
    markersize=4, alpha=0.8
)
ax.plot(
    rounds, fl_history['h3_train_loss'],
    marker='^', label='Hospital 3 (BreakHis)',
    linewidth=2, color=COLOR_H3,
    markersize=4, alpha=0.8
)

ax.set_xlabel('FL Round', fontsize=12, fontweight='bold')
ax.set_ylabel('Training Loss', fontsize=12, fontweight='bold')
ax.set_title(
    'Federated Learning (FedAvg) — Local Training Loss Over Rounds\n'
    'Each hospital trains locally before shared head aggregation',
    fontsize=12, fontweight='bold'
)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    'results/federated/fl_training_loss.png',
    dpi=300, bbox_inches='tight'
)
plt.close()
print("    fl_training_loss.png")

# PLOT 5 — Combined dashboard
# All metrics in one figure for thesis
fig = plt.figure(figsize=(16, 12))
fig.suptitle(
    f'Federated Learning (FedAvg) — Complete Results Dashboard\n'
    f'Shared Embedding Space ({EMBEDDING_DIM}-dim) | '
    f'3 Hospitals | {num_fl_rounds} Rounds',
    fontsize=13, fontweight='bold'
)

gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)

# Top left — Accuracy
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(rounds, fl_history['h1_test_acc'],
         color=COLOR_H1, marker='o', markersize=3,
         linewidth=1.5, label='H1 WDBC')
ax1.plot(rounds, fl_history['h2_test_acc'],
         color=COLOR_H2, marker='s', markersize=3,
         linewidth=1.5, label='H2 Coimbra')
ax1.plot(rounds, fl_history['h3_test_acc'],
         color=COLOR_H3, marker='^', markersize=3,
         linewidth=1.5, label='H3 BreakHis')
ax1.plot(rounds, fl_history['avg_test_acc'],
         color=COLOR_AVG, marker='D', markersize=3,
         linewidth=2, linestyle='--', label='Avg')
ax1.set_title('Test Accuracy', fontweight='bold')
ax1.set_xlabel('FL Round')
ax1.set_ylabel('Accuracy')
ax1.set_ylim([0.0, 1.05])
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Top right — F1 Score
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(rounds, fl_history['h1_test_f1'],
         color=COLOR_H1, marker='o', markersize=3, linewidth=1.5)
ax2.plot(rounds, fl_history['h2_test_f1'],
         color=COLOR_H2, marker='s', markersize=3, linewidth=1.5)
ax2.plot(rounds, fl_history['h3_test_f1'],
         color=COLOR_H3, marker='^', markersize=3, linewidth=1.5)
ax2.plot(rounds, fl_history['avg_test_f1'],
         color=COLOR_AVG, marker='D', markersize=3,
         linewidth=2, linestyle='--')
ax2.set_title('F1-Score', fontweight='bold')
ax2.set_xlabel('FL Round')
ax2.set_ylabel('F1-Score')
ax2.set_ylim([0.0, 1.05])
ax2.grid(True, alpha=0.3)

# Bottom left — AUC-ROC
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(rounds, fl_history['h1_test_auc'],
         color=COLOR_H1, marker='o', markersize=3, linewidth=1.5)
ax3.plot(rounds, fl_history['h2_test_auc'],
         color=COLOR_H2, marker='s', markersize=3, linewidth=1.5)
ax3.plot(rounds, fl_history['h3_test_auc'],
         color=COLOR_H3, marker='^', markersize=3, linewidth=1.5)
ax3.plot(rounds, fl_history['avg_test_auc'],
         color=COLOR_AVG, marker='D', markersize=3,
         linewidth=2, linestyle='--')
ax3.axhline(y=0.5, color='gray', linestyle='--',
            alpha=0.5, label='Random')
ax3.set_title('AUC-ROC', fontweight='bold')
ax3.set_xlabel('FL Round')
ax3.set_ylabel('AUC-ROC')
ax3.set_ylim([0.0, 1.05])
ax3.grid(True, alpha=0.3)

# Bottom right — Training Loss
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(rounds, fl_history['h1_train_loss'],
         color=COLOR_H1, marker='o', markersize=3,
         linewidth=1.5, label='H1 WDBC')
ax4.plot(rounds, fl_history['h2_train_loss'],
         color=COLOR_H2, marker='s', markersize=3,
         linewidth=1.5, label='H2 Coimbra')
ax4.plot(rounds, fl_history['h3_train_loss'],
         color=COLOR_H3, marker='^', markersize=3,
         linewidth=1.5, label='H3 BreakHis')
ax4.set_title('Local Training Loss', fontweight='bold')
ax4.set_xlabel('FL Round')
ax4.set_ylabel('Loss')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

plt.savefig(
    'results/federated/fl_dashboard.png',
    dpi=300, bbox_inches='tight'
)
plt.close()
print("    fl_dashboard.png")

# Final summary
print(f"\n{'='*55}")
print(f"  PLOTS SAVED SUMMARY")
print(f"{'='*55}")
print(f"  fl_test_accuracy.png  : Accuracy per hospital + avg")
print(f"  fl_f1_score.png       : F1 per hospital + avg")
print(f"  fl_auc_roc.png        : AUC per hospital + avg")
print(f"  fl_training_loss.png  : Local training loss")
print(f"  fl_dashboard.png      : Combined thesis dashboard")
print(f"  {'─'*53}")
print(f"  Save directory        : results/federated/")
print(f"  Resolution            : 300 DPI")
print(f"{'='*55}")


  PHASE 4 — GENERATING FEDERATED LEARNING PLOTS
    fl_test_accuracy.png
    fl_f1_score.png
    fl_auc_roc.png
    fl_training_loss.png
    fl_dashboard.png

  PLOTS SAVED SUMMARY
  fl_test_accuracy.png  : Accuracy per hospital + avg
  fl_f1_score.png       : F1 per hospital + avg
  fl_auc_roc.png        : AUC per hospital + avg
  fl_training_loss.png  : Local training loss
  fl_dashboard.png      : Combined thesis dashboard
  ─────────────────────────────────────────────────────
  Save directory        : results/federated/
  Resolution            : 300 DPI


### 8. Compare with Local Baselines

In [10]:
import json
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs('results/federated', exist_ok=True)

print("\n" + "="*70)
print("  PHASE 4 — COMPARING FEDERATED VS LOCAL PERFORMANCE")
print("="*70)

# STEP 1 — Loading local baseline metrics from Phase 3
# Use full metrics including embedding analysis
print("\n Loading Phase 3 local baseline metrics...")

with open(
    'results/local_training/hospital1_metrics.json',
    'r', encoding='utf-8'
) as f:
    local_h1 = json.load(f)

with open(
    'results/local_training/hospital2_metrics.json',
    'r', encoding='utf-8'
) as f:
    local_h2 = json.load(f)

with open(
    'results/local_training/hospital3_metrics.json',
    'r', encoding='utf-8'
) as f:
    local_h3 = json.load(f)

print("    Hospital 1 local metrics loaded")
print("    Hospital 2 local metrics loaded")
print("    Hospital 3 local metrics loaded")

# STEP 2 — Getting final federated metrics
# Use best round metrics not just last round
# Best round = round with highest weighted average accuracy
best_round_idx = np.argmax(fl_history['avg_test_acc'])

print(f"\n Using best FL round: {fl_history['round'][best_round_idx]}"
      f" (highest weighted avg accuracy)")

final_h1 = {
    'accuracy': fl_history['h1_test_acc'][best_round_idx],
    'f1'      : fl_history['h1_test_f1'][best_round_idx],
    'auc_roc' : fl_history['h1_test_auc'][best_round_idx]
}
final_h2 = {
    'accuracy': fl_history['h2_test_acc'][best_round_idx],
    'f1'      : fl_history['h2_test_f1'][best_round_idx],
    'auc_roc' : fl_history['h2_test_auc'][best_round_idx]
}
final_h3 = {
    'accuracy': fl_history['h3_test_acc'][best_round_idx],
    'f1'      : fl_history['h3_test_f1'][best_round_idx],
    'auc_roc' : fl_history['h3_test_auc'][best_round_idx]
}

# STEP 3 — Preparing comparison data
hospitals     = [
    'Hospital 1\n(WDBC)',
    'Hospital 2\n(Coimbra)',
    'Hospital 3\n(BreakHis)'
]
metrics_names = ['Accuracy', 'F1-Score', 'AUC-ROC']
metrics_keys  = ['accuracy', 'f1', 'auc_roc']

local_metrics = [local_h1, local_h2, local_h3]
final_metrics = [final_h1, final_h2, final_h3]

local_data = [
    [local_h1[k], local_h2[k], local_h3[k]]
    for k in metrics_keys
]
federated_data = [
    [final_h1[k], final_h2[k], final_h3[k]]
    for k in metrics_keys
]

# STEP 4 — Bar Chart Comparison Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle(
    f'Local Training vs Federated Learning (FedAvg) — Performance Comparison\n'
    f'Encoder (private) + SharedHead (federated) | '
    f'Embedding dim = {EMBEDDING_DIM} | '
    f'Best FL round = {fl_history["round"][best_round_idx]}',
    fontsize=12, fontweight='bold', y=1.02
)

COLOR_LOCAL = '#E74C3C'   # red   — local only
COLOR_FED   = '#3498DB'   # blue  — federated

x     = np.arange(len(hospitals))
width = 0.35

for idx, ax in enumerate(axes):
    bars_local = ax.bar(
        x - width/2,
        local_data[idx],
        width,
        label='Local Only (Phase 3)',
        color=COLOR_LOCAL,
        alpha=0.85,
        edgecolor='white',
        linewidth=1.2
    )
    bars_fed = ax.bar(
        x + width/2,
        federated_data[idx],
        width,
        label=f'Federated (FedAvg)',
        color=COLOR_FED,
        alpha=0.85,
        edgecolor='white',
        linewidth=1.2
    )

    # Value labels on bars
    for bars in [bars_local, bars_fed]:
        for bar in bars:
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2.,
                height + 0.01,
                f'{height:.3f}',
                ha='center', va='bottom',
                fontsize=8.5, fontweight='bold'
            )

    # Improvement arrows — show delta for each hospital
    for i in range(len(hospitals)):
        local_val = local_data[idx][i]
        fed_val   = federated_data[idx][i]
        delta     = fed_val - local_val
        color     = '#27AE60' if delta >= 0 else '#C0392B'
        symbol    = '▲' if delta >= 0 else '▼'
        ax.text(
            x[i],
            max(local_val, fed_val) + 0.06,
            f'{symbol}{abs(delta):.3f}',
            ha='center', va='bottom',
            fontsize=8, color=color,
            fontweight='bold'
        )

    ax.set_ylabel(metrics_names[idx], fontsize=12, fontweight='bold')
    ax.set_title(
        f'{metrics_names[idx]} Comparison',
        fontsize=12, fontweight='bold'
    )
    ax.set_xticks(x)
    ax.set_xticklabels(hospitals, fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0.0, 1.15])

plt.tight_layout()
plt.savefig(
    'results/federated/local_vs_federated_comparison.png',
    dpi=300, bbox_inches='tight'
)
plt.close()
print("\n    local_vs_federated_comparison.png")

# STEP 5 — Per-hospital detailed bar chart
# One figure per hospital showing all metrics side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'Per-Hospital Detailed Metric Comparison\n'
    'Local Only vs Federated Learning (FedAvg)',
    fontsize=13, fontweight='bold'
)

hospital_names_full = [
    'Hospital 1 — WDBC\n(Cell Nucleus Features)',
    'Hospital 2 — Coimbra\n(Blood Biomarkers)',
    'Hospital 3 — BreakHis\n(Histopathology Images)'
]
hospital_colors = ['#2ECC71', '#E74C3C', '#3498DB']

for h_idx, ax in enumerate(axes):
    x_pos     = np.arange(len(metrics_names))
    local_vals = [local_data[m][h_idx] for m in range(3)]
    fed_vals   = [federated_data[m][h_idx] for m in range(3)]

    bars_l = ax.bar(
        x_pos - width/2, local_vals, width,
        label='Local Only',
        color='#BDC3C7', alpha=0.9,
        edgecolor='white', linewidth=1.2
    )
    bars_f = ax.bar(
        x_pos + width/2, fed_vals, width,
        label='Federated',
        color=hospital_colors[h_idx], alpha=0.9,
        edgecolor='white', linewidth=1.2
    )

    for bars in [bars_l, bars_f]:
        for bar in bars:
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2.,
                height + 0.01,
                f'{height:.3f}',
                ha='center', va='bottom',
                fontsize=9, fontweight='bold'
            )

    ax.set_title(
        hospital_names_full[h_idx],
        fontsize=10, fontweight='bold'
    )
    ax.set_xticks(x_pos)
    ax.set_xticklabels(metrics_names, fontsize=10)
    ax.set_ylabel('Score', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0.0, 1.15])

plt.tight_layout()
plt.savefig(
    'results/federated/per_hospital_comparison.png',
    dpi=300, bbox_inches='tight'
)
plt.close()
print("    per_hospital_comparison.png")

# STEP 6 — Numerical comparison table
print(f"\n{'='*70}")
print(f"  NUMERICAL COMPARISON — LOCAL vs FEDERATED")
print(f"{'='*70}")
print(f"  Best FL round used : {fl_history['round'][best_round_idx]}")
print(f"  {'─'*68}")

hospital_names_short = [
    'Hospital 1 (WDBC)',
    'Hospital 2 (Coimbra)',
    'Hospital 3 (BreakHis)'
]

for i, name in enumerate(hospital_names_short):
    print(f"\n  {name}:")
    print(f"  {'Metric':<12} {'Local':>10} {'Federated':>12} "
          f"{'Delta':>10} {'Better?':>10}")
    print(f"  {'─'*56}")

    for m_idx, metric in enumerate(metrics_names):
        local_val = local_data[m_idx][i]
        fed_val   = federated_data[m_idx][i]
        delta     = fed_val - local_val
        better    = ' FL' if delta > 0 else ('Same' if delta == 0 else 'Local')

        print(f"  {metric:<12} {local_val:>10.4f} {fed_val:>12.4f} "
              f"{delta:>+10.4f} {better:>10}")

# STEP 7 — Overall improvement summary
print(f"\n{'='*70}")
print(f"  OVERALL IMPROVEMENT SUMMARY")
print(f"{'='*70}")

for m_idx, metric in enumerate(metrics_names):
    local_avg = np.mean(local_data[m_idx])
    fed_avg   = np.mean(federated_data[m_idx])
    delta     = fed_avg - local_avg
    symbol    = '▲' if delta >= 0 else '▼'
    print(f"  {metric:<12} "
          f"Local avg={local_avg:.4f}  "
          f"FL avg={fed_avg:.4f}  "
          f"Delta={symbol}{abs(delta):.4f}")

# STEP 8 — Saving comparison results as JSON
comparison_results = {
    'best_fl_round'    : int(fl_history['round'][best_round_idx]),
    'fl_algorithm'     : 'FedAvg',
    'embedding_dim'    : EMBEDDING_DIM,
    'hospital1': {
        'local'     : {k: float(local_h1[k]) for k in metrics_keys},
        'federated' : {k: float(final_h1[k]) for k in metrics_keys},
        'delta'     : {
            k: float(final_h1[k] - local_h1[k])
            for k in metrics_keys
        }
    },
    'hospital2': {
        'local'     : {k: float(local_h2[k]) for k in metrics_keys},
        'federated' : {k: float(final_h2[k]) for k in metrics_keys},
        'delta'     : {
            k: float(final_h2[k] - local_h2[k])
            for k in metrics_keys
        }
    },
    'hospital3': {
        'local'     : {k: float(local_h3[k]) for k in metrics_keys},
        'federated' : {k: float(final_h3[k]) for k in metrics_keys},
        'delta'     : {
            k: float(final_h3[k] - local_h3[k])
            for k in metrics_keys
        }
    }
}

with open(
    'results/federated/local_vs_federated_comparison.json',
    'w', encoding='utf-8'
) as f:
    json.dump(comparison_results, f, indent=4)

print(f"\n    Comparison results saved: "
      f"results/federated/local_vs_federated_comparison.json")

# STEP 9 — Final summary
print(f"\n{'='*55}")
print(f"  COMPARISON SUMMARY")
print(f"{'='*55}")
print(f"  Plots saved:")
print(f"    local_vs_federated_comparison.png")
print(f"    per_hospital_comparison.png")
print(f"  Data saved:")
print(f"    local_vs_federated_comparison.json")
print(f"  Best FL round    : {fl_history['round'][best_round_idx]}")
print(f"  Algorithm        : FedAvg")
print(f"  Embedding dim    : {EMBEDDING_DIM}")
print(f"  Next Step        : Phase 5 — Differential Privacy")
print(f"{'='*55}")


  PHASE 4 — COMPARING FEDERATED VS LOCAL PERFORMANCE

 Loading Phase 3 local baseline metrics...
    Hospital 1 local metrics loaded
    Hospital 2 local metrics loaded
    Hospital 3 local metrics loaded

 Using best FL round: 17 (highest weighted avg accuracy)

    local_vs_federated_comparison.png
    per_hospital_comparison.png

  NUMERICAL COMPARISON — LOCAL vs FEDERATED
  Best FL round used : 17
  ────────────────────────────────────────────────────────────────────

  Hospital 1 (WDBC):
  Metric            Local    Federated      Delta    Better?
  ────────────────────────────────────────────────────────
  Accuracy         0.9737       0.9825    +0.0088         FL
  F1-Score         0.9639       0.9762    +0.0123         FL
  AUC-ROC          0.9967       0.9947    -0.0020      Local

  Hospital 2 (Coimbra):
  Metric            Local    Federated      Delta    Better?
  ────────────────────────────────────────────────────────
  Accuracy         0.7083       0.7083    +0.0000    

## FINAL SUMMARY

### 9. Phase 4 Summary

In [11]:
import os
import json

os.makedirs('results/federated', exist_ok=True)

# ============================================================
# Compute weighted average metrics for summary
# ============================================================
total_samples = sum(dataset_sizes)
weights       = [s / total_samples for s in dataset_sizes]

local_avg_acc = sum(
    w * m['accuracy']
    for w, m in zip(weights, [local_h1, local_h2, local_h3])
)
local_avg_f1 = sum(
    w * m['f1']
    for w, m in zip(weights, [local_h1, local_h2, local_h3])
)
local_avg_auc = sum(
    w * m['auc_roc']
    for w, m in zip(weights, [local_h1, local_h2, local_h3])
)

fed_avg_acc = fl_history['avg_test_acc'][best_round_idx]
fed_avg_f1  = fl_history['avg_test_f1'][best_round_idx]
fed_avg_auc = fl_history['avg_test_auc'][best_round_idx]

# Determine if FL improved over local for each hospital
h1_improved = final_h1['accuracy'] >= local_h1['accuracy']
h2_improved = final_h2['accuracy'] >= local_h2['accuracy']
h3_improved = final_h3['accuracy'] >= local_h3['accuracy']
all_improved = all([h1_improved, h2_improved, h3_improved])

# FL params exchanged per round
fl_params_per_round = sum(
    p.numel() for p in model_h1.shared_head.parameters()
)

print("\n" + "="*70)
print("  PHASE 4 COMPLETE — FEDERATED LEARNING WITHOUT DIFFERENTIAL PRIVACY")
print("="*70)

summary = f"""
==============================================================
  FEDERATED LEARNING RESULTS (WITHOUT DIFFERENTIAL PRIVACY)
==============================================================

** Configuration:
   Algorithm          : Weighted FedAvg
   FL rounds          : {num_fl_rounds}
   Local epochs/round : {local_epochs}
   Privacy            : None (baseline for Phase 5 comparison)
   Embedding dim      : {EMBEDDING_DIM} (shared across all hospitals)
   FL params/round    : {fl_params_per_round:,} (shared head only)
   Architecture       : Encoder (private) + SharedHead (federated)
   Best round         : {fl_history['round'][best_round_idx]}

** Dataset Weights (FedAvg contribution):
   Hospital 1 (WDBC)    : {dataset_sizes[0]:,} samples ({weights[0]*100:.1f}%)
   Hospital 2 (Coimbra) : {dataset_sizes[1]:,} samples ({weights[1]*100:.1f}%)
   Hospital 3 (BreakHis): {dataset_sizes[2]:,} samples ({weights[2]*100:.1f}%)
   Total                : {total_samples:,} samples

==============================================================
  PERFORMANCE COMPARISON — LOCAL vs FEDERATED
==============================================================

** Hospital 1 (WDBC - Cell Nucleus Features):
   Metric      Local Only    Federated     Delta
   Accuracy    {local_h1['accuracy']:.4f}        {final_h1['accuracy']:.4f}        {final_h1['accuracy']-local_h1['accuracy']:+.4f}
   F1-Score    {local_h1['f1']:.4f}        {final_h1['f1']:.4f}        {final_h1['f1']-local_h1['f1']:+.4f}
   AUC-ROC     {local_h1['auc_roc']:.4f}        {final_h1['auc_roc']:.4f}        {final_h1['auc_roc']-local_h1['auc_roc']:+.4f}
   FL improved : {'Yes' if h1_improved else 'No (local was stronger)'}

** Hospital 2 (Coimbra - Blood Biomarkers):
   Metric      Local Only    Federated     Delta
   Accuracy    {local_h2['accuracy']:.4f}        {final_h2['accuracy']:.4f}        {final_h2['accuracy']-local_h2['accuracy']:+.4f}
   F1-Score    {local_h2['f1']:.4f}        {final_h2['f1']:.4f}        {final_h2['f1']-local_h2['f1']:+.4f}
   AUC-ROC     {local_h2['auc_roc']:.4f}        {final_h2['auc_roc']:.4f}        {final_h2['auc_roc']-local_h2['auc_roc']:+.4f}
   FL improved : {'Yes' if h2_improved else 'No (local was stronger)'}
   Note        : Only {len(fl_history['h2_test_acc'])} rounds — small dataset (92 samples)

** Hospital 3 (BreakHis - Histopathology Images):
   Metric      Local Only    Federated     Delta
   Accuracy    {local_h3['accuracy']:.4f}        {final_h3['accuracy']:.4f}        {final_h3['accuracy']-local_h3['accuracy']:+.4f}
   F1-Score    {local_h3['f1']:.4f}        {final_h3['f1']:.4f}        {final_h3['f1']-local_h3['f1']:+.4f}
   AUC-ROC     {local_h3['auc_roc']:.4f}        {final_h3['auc_roc']:.4f}        {final_h3['auc_roc']-local_h3['auc_roc']:+.4f}
   FL improved : {'Yes' if h3_improved else 'No (local was stronger)'}

==============================================================
  WEIGHTED AVERAGE ACROSS ALL HOSPITALS
==============================================================

   Metric      Local Only    Federated     Delta
   Accuracy    {local_avg_acc:.4f}        {fed_avg_acc:.4f}        {fed_avg_acc-local_avg_acc:+.4f}
   F1-Score    {local_avg_f1:.4f}        {fed_avg_f1:.4f}        {fed_avg_f1-local_avg_f1:+.4f}
   AUC-ROC     {local_avg_auc:.4f}        {fed_avg_auc:.4f}        {fed_avg_auc-local_avg_auc:+.4f}

==============================================================
  KEY INSIGHTS
==============================================================

   FL outcome       : {'Improved all hospitals' if all_improved else 'Mixed results — not all hospitals improved'}
   Privacy preserved: Yes — no raw data left any hospital
   Shared component : Only {fl_params_per_round:,} shared head params per round
   Private kept     : Encoders (H1: {sum(p.numel() for p in model_h1.encoder.parameters()):,} | H2: {sum(p.numel() for p in model_h2.encoder.parameters()):,} | H3: {sum(p.numel() for p in model_h3.encoder.projection.parameters()):,} params)
   FedAvg valid     : Yes — shared 64-dim embedding space confirmed
   H2 note          : Small dataset (92 samples) limits FL contribution
                      Expected behavior — FL addresses this in Phase 5

==============================================================
  SAVED FILES
==============================================================

   Models (full):
   - models/federated/hospital1_federated.pth
   - models/federated/hospital2_federated.pth
   - models/federated/hospital3_federated.pth

   Encoders (private):
   - models/federated/hospital1_encoder_federated.pth
   - models/federated/hospital2_encoder_federated.pth
   - models/federated/hospital3_encoder_federated.pth

   Global shared head:
   - models/federated/final_global_shared_head.pth
   - models/federated/best_global_shared_head.pth

   Results:
   - results/federated/fl_history.json
   - results/federated/fl_test_accuracy.png
   - results/federated/fl_f1_score.png
   - results/federated/fl_auc_roc.png
   - results/federated/fl_training_loss.png
   - results/federated/fl_dashboard.png
   - results/federated/local_vs_federated_comparison.png
   - results/federated/per_hospital_comparison.png
   - results/federated/local_vs_federated_comparison.json

   Phase summary:
   - results/federated/phase4_summary.txt

Next Step: Phase 5 - Add Differential Privacy
           Compare privacy vs accuracy tradeoff
           Test epsilon levels: 10, 1, 0.1
"""

print(summary)
print("="*70)

# ============================================================
# Save summary with utf-8 encoding
# ============================================================
try:
    with open(
        'results/federated/phase4_summary.txt',
        'w',
        encoding='utf-8'
    ) as f:
        f.write(summary)

    # Verify saved correctly
    with open(
        'results/federated/phase4_summary.txt',
        'r',
        encoding='utf-8'
    ) as f:
        saved = f.read()

    assert 'Hospital 1' in saved
    assert 'Hospital 2' in saved
    assert 'Hospital 3' in saved
    assert 'FedAvg'     in saved

    print(" Summary saved and verified: "
          "results/federated/phase4_summary.txt")

except Exception as e:
    print(f" Summary save failed: {e}")


  PHASE 4 COMPLETE — FEDERATED LEARNING WITHOUT DIFFERENTIAL PRIVACY

  FEDERATED LEARNING RESULTS (WITHOUT DIFFERENTIAL PRIVACY)

** Configuration:
   Algorithm          : Weighted FedAvg
   FL rounds          : 25
   Local epochs/round : 2
   Privacy            : None (baseline for Phase 5 comparison)
   Embedding dim      : 64 (shared across all hospitals)
   FL params/round    : 2,177 (shared head only)
   Architecture       : Encoder (private) + SharedHead (federated)
   Best round         : 17

** Dataset Weights (FedAvg contribution):
   Hospital 1 (WDBC)    : 570 samples (21.5%)
   Hospital 2 (Coimbra) : 92 samples (3.5%)
   Hospital 3 (BreakHis): 1,990 samples (75.0%)
   Total                : 2,652 samples

  PERFORMANCE COMPARISON — LOCAL vs FEDERATED

** Hospital 1 (WDBC - Cell Nucleus Features):
   Metric      Local Only    Federated     Delta
   Accuracy    0.9737        0.9825        +0.0088
   F1-Score    0.9639        0.9762        +0.0123
   AUC-ROC     0.9967       